### Week 1 - Picking Best Single Feature (Income - Spending)

In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score

In [106]:
# Reading data
acct_df = pd.read_parquet("data/q2-ucsd-acctDF.pqt")
cons_df = pd.read_parquet("data/q2-ucsd-consDF.pqt")
txn_df = pd.read_parquet("data/q2-ucsd-trxnDF.pqt")
cat_map_df = pd.read_csv("data/q2-ucsd-cat-map.csv")

### Visualize each dataframe

In [107]:
acct_df

,prism_consumer_id,prism_account_id,account_type,balance_date,balance
0,3023,0,SAVINGS,2021-08-31,90.57
1,3023,1,CHECKING,2021-08-31,225.95
2,4416,2,SAVINGS,2022-03-31,15157.17
3,4416,3,CHECKING,2022-03-31,66.42
4,4227,4,CHECKING,2021-07-31,7042.90
...,...,...,...,...,...
24461,11500,24461,CHECKING,2022-03-27,732.75
24462,11615,24462,SAVINGS,2022-03-30,5.00
24463,11615,24463,CHECKING,2022-03-30,1956.46
24464,12210,24464,CHECKING,2022-03-28,2701.51


In [56]:
train_df = cons_df[:12000]
train_df

,prism_consumer_id,evaluation_date,credit_score,DQ_TARGET
0,0,2021-09-01,726.0,0.0
1,1,2021-07-01,626.0,0.0
2,2,2021-05-01,680.0,0.0
3,3,2021-03-01,734.0,0.0
4,4,2021-10-01,676.0,0.0
...,...,...,...,...
11995,11995,2022-03-19,623.0,0.0
11996,11996,2022-03-18,617.0,0.0
11997,11997,2022-02-25,646.0,0.0
11998,11998,2022-02-09,773.0,0.0


In [4]:
len(cons_df['prism_consumer_id'].unique())

15000

In [5]:
txn_df

,prism_consumer_id,prism_transaction_id,category,amount,credit_or_debit,posted_date
0,3023,0,4,0.05,CREDIT,2021-04-16
1,3023,1,12,481.56,CREDIT,2021-04-30
2,3023,2,4,0.05,CREDIT,2021-05-16
3,3023,3,4,0.07,CREDIT,2021-06-16
4,3023,4,4,0.06,CREDIT,2021-07-16
...,...,...,...,...,...,...
6407316,10533,6405304,31,4.96,DEBIT,2022-03-11
6407317,10533,6405305,12,63.48,DEBIT,2022-03-30
6407318,10533,6405306,12,53.99,DEBIT,2022-03-30
6407319,10533,6405307,12,175.98,DEBIT,2022-03-31


In [6]:
cat_map_df


,category_id,category
0,0,SELF_TRANSFER
1,1,EXTERNAL_TRANSFER
2,2,DEPOSIT
3,3,PAYCHECK
4,4,MISCELLANEOUS
5,5,PAYCHECK_PLACEHOLDER
6,6,REFUND
7,7,INVESTMENT_INCOME
8,8,OTHER_BENEFITS
9,9,UNEMPLOYMENT_BENEFITS


### Data Preprocessing

In [7]:
# Convert Category Mappings to a dictionary

cat_map = dict()

for i in range(cat_map_df.shape[0]):
    cat_map[i] = cat_map_df['category'].iloc[i]

cat_map

{0: 'SELF_TRANSFER',
 1: 'EXTERNAL_TRANSFER',
 2: 'DEPOSIT',
 3: 'PAYCHECK',
 4: 'MISCELLANEOUS',
 5: 'PAYCHECK_PLACEHOLDER',
 6: 'REFUND',
 7: 'INVESTMENT_INCOME',
 8: 'OTHER_BENEFITS',
 9: 'UNEMPLOYMENT_BENEFITS',
 10: 'SMALL_DOLLAR_ADVANCE',
 11: 'TAX',
 12: 'LOAN',
 13: 'INSURANCE',
 14: 'FOOD_AND_BEVERAGES',
 15: 'UNCATEGORIZED',
 16: 'GENERAL_MERCHANDISE',
 17: 'AUTOMOTIVE',
 18: 'GROCERIES',
 19: 'ATM_CASH',
 20: 'ENTERTAINMENT',
 21: 'TRAVEL',
 22: 'ESSENTIAL_SERVICES',
 23: 'ACCOUNT_FEES',
 24: 'HOME_IMPROVEMENT',
 25: 'OVERDRAFT',
 26: 'CREDIT_CARD_PAYMENT',
 27: 'HEALTHCARE_MEDICAL',
 28: 'PETS',
 29: 'EDUCATION',
 30: 'GIFTS_DONATIONS',
 31: 'BILLS_UTILITIES',
 32: 'MORTGAGE',
 33: 'CHILD_DEPENDENTS',
 34: 'RENT',
 35: 'BNPL',
 36: 'AUTO_LOAN',
 37: 'BANKING_CATCH_ALL',
 38: 'DEBT',
 39: 'FITNESS',
 40: 'TRANSPORATION',
 41: 'LEGAL',
 42: 'GOVERNMENT_SERVICES',
 43: 'RISK_CATCH_ALL',
 44: 'RTO_LTO',
 45: 'INVESTMENT',
 46: 'GAMBLING',
 47: 'CORPORATE_PAYMENTS',
 48: 'TIME_O

In [8]:
# Map all categories into the dataframe

txn_df['category'] = txn_df['category'].apply(lambda x: cat_map[x])

In [9]:
# Divide the transaction dataset into debit and credit

deb_txn = txn_df[txn_df['credit_or_debit'] == "DEBIT"]
cre_txn = txn_df[txn_df['credit_or_debit'] == "CREDIT"]

In [10]:
sum(deb_txn['amount'] < 0)

0

In [11]:
cre_txn[cre_txn['amount'] < 0]['category'].value_counts()

Series([], Name: count, dtype: int64)

In [12]:
cre_txn[cre_txn['amount'] > 0]['category'].value_counts()

category
SELF_TRANSFER            342392
EXTERNAL_TRANSFER        216021
PAYCHECK                 180208
DEPOSIT                  114156
MISCELLANEOUS            112397
REFUND                    60841
INVESTMENT_INCOME         21813
LOAN                      13585
TAX                       10262
UNEMPLOYMENT_BENEFITS      7521
INVESTMENT                 6684
PENSION                    6032
TIME_OR_STUFF              5660
OTHER_BENEFITS             5513
INSURANCE                  1432
GAMBLING                    424
CORPORATE_PAYMENTS           22
Name: count, dtype: int64

In [13]:
txn_df['category'].unique()

array(['MISCELLANEOUS', 'LOAN', 'EXTERNAL_TRANSFER', 'DEPOSIT',
       'SELF_TRANSFER', 'INVESTMENT', 'PAYCHECK', 'REFUND',
       'ENTERTAINMENT', 'FOOD_AND_BEVERAGES', 'GROCERIES', 'FITNESS',
       'GENERAL_MERCHANDISE', 'HEALTHCARE_MEDICAL', 'GAMBLING',
       'GIFTS_DONATIONS', 'CREDIT_CARD_PAYMENT', 'AUTOMOTIVE',
       'HOME_IMPROVEMENT', 'ATM_CASH', 'TRANSPORATION', 'PENSION',
       'INSURANCE', 'TAX', 'ACCOUNT_FEES', 'RENT', 'BILLS_UTILITIES',
       'ESSENTIAL_SERVICES', 'EDUCATION', 'TRAVEL', 'PETS', 'MORTGAGE',
       'BANKING_CATCH_ALL', 'DEBT', 'AUTO_LOAN', 'BNPL',
       'GOVERNMENT_SERVICES', 'CORPORATE_PAYMENTS', 'LEGAL',
       'RISK_CATCH_ALL', 'OTHER_BENEFITS', 'TIME_OR_STUFF',
       'UNEMPLOYMENT_BENEFITS', 'CHILD_DEPENDENTS', 'RTO_LTO',
       'INVESTMENT_INCOME', 'OVERDRAFT'], dtype=object)

### Feature Engineering

To find the best feature, I am trying to aggregate the transaction df on consumer-level and get a feature (net monthly income) that represent "estimated monthly income" - "estimated monthly spending".

In [120]:
# Aggregate transaction amount based on individual consumer and year-month pair

txn_df['year_month'] = pd.to_datetime(txn_df['posted_date']).dt.to_period('M')

txn_aggregated = txn_df.groupby(['prism_consumer_id', 'year_month']).agg(
    monthly_income=('amount', lambda x: x[txn_df.loc[x.index, 'credit_or_debit'] == 'CREDIT'].sum()),
    monthly_spending=('amount', lambda x: x[txn_df.loc[x.index, 'credit_or_debit'] == 'DEBIT'].sum())
).reset_index()

txn_aggregated

,prism_consumer_id,year_month,monthly_income,monthly_spending
0,0,2021-03,2400.69,1999.35
1,0,2021-04,3210.66,2379.93
2,0,2021-05,967.00,765.50
3,0,2021-06,1068.13,3652.37
4,0,2021-07,2204.83,2276.21
...,...,...,...,...
89548,9998,2023-10,5140.21,5124.35
89549,9999,2023-05,7664.45,4488.76
89550,9999,2023-06,12576.78,12813.20
89551,9999,2023-07,10370.51,11658.85


In [15]:
txn_aggregated['monthly_net_income'] = txn_aggregated.apply(lambda row: row['monthly_income'] - row['monthly_spending'], axis = 1)
txn_aggregated

,prism_consumer_id,year_month,monthly_income,monthly_spending,monthly_net_income
0,0,2021-03,2400.69,1999.35,401.34
1,0,2021-04,3210.66,2379.93,830.73
2,0,2021-05,967.00,765.50,201.50
3,0,2021-06,1068.13,3652.37,-2584.24
4,0,2021-07,2204.83,2276.21,-71.38
...,...,...,...,...,...
89548,9998,2023-10,5140.21,5124.35,15.86
89549,9999,2023-05,7664.45,4488.76,3175.69
89550,9999,2023-06,12576.78,12813.20,-236.42
89551,9999,2023-07,10370.51,11658.85,-1288.34


In [16]:
txn_avg_monthly = txn_aggregated.groupby('prism_consumer_id').agg(
    average_nmi=('monthly_net_income', 'mean')
).reset_index()

txn_avg_monthly

,prism_consumer_id,average_nmi
0,0,-74.512857
1,1,257.918571
2,10,-170.005714
3,100,-750.961667
4,1000,62.582857
...,...,...
14487,9995,-30.387500
14488,9996,2.285000
14489,9997,125.797500
14490,9998,-277.700000


In [ ]:
def zscoring_normalize(df, column):
    # Calculate the mean and standard deviation of the entire balance column
    mean = df[column].mean()
    std = df[column].std()

    # Z-score normalization
    df[column + '_z_score'] = (df[column] - mean) / std

In [76]:
txn_avg_monthly

zscoring_normalize(txn_avg_monthly, 'average_nmi')

,prism_consumer_id,average_nmi,average_nmi_z_score
0,0,-74.512857,-0.065380
1,1,257.918571,-0.029751
2,10,-170.005714,-0.075614
3,100,-750.961667,-0.137879
4,1000,62.582857,-0.050687
...,...,...,...
14487,9995,-30.387500,-0.060651
14488,9996,2.285000,-0.057149
14489,9997,125.797500,-0.043912
14490,9998,-277.700000,-0.087157


In [17]:
txn_cons = set(txn_avg_monthly['prism_consumer_id'].unique())
cons_cons = set(cons_df['prism_consumer_id'].unique())

len(cons_cons - txn_cons)

508

In [18]:
cons_cons - txn_cons

{'10234',
 '11008',
 '11504',
 '11655',
 '11918',
 '12101',
 '12780',
 '12932',
 '13479',
 '13607',
 '13750',
 '14056',
 '14163',
 '14206',
 '14435',
 '14749',
 '14916',
 '5003',
 '5007',
 '5024',
 '5036',
 '5044',
 '5082',
 '5084',
 '5102',
 '5107',
 '5109',
 '5115',
 '5125',
 '5129',
 '5130',
 '5149',
 '5166',
 '5177',
 '5179',
 '5186',
 '5200',
 '5202',
 '5206',
 '5228',
 '5247',
 '5272',
 '5283',
 '5284',
 '5295',
 '5298',
 '5301',
 '5302',
 '5308',
 '5311',
 '5315',
 '5319',
 '5332',
 '5335',
 '5351',
 '5366',
 '5381',
 '5388',
 '5403',
 '5423',
 '5437',
 '5451',
 '5469',
 '5496',
 '5506',
 '5513',
 '5524',
 '5528',
 '5543',
 '5558',
 '5588',
 '5600',
 '5608',
 '5612',
 '5614',
 '5634',
 '5649',
 '5660',
 '5670',
 '5671',
 '5690',
 '5697',
 '5714',
 '5716',
 '5722',
 '5732',
 '5755',
 '5758',
 '5773',
 '5788',
 '5814',
 '5829',
 '5830',
 '5844',
 '5845',
 '5866',
 '5868',
 '5878',
 '5890',
 '5902',
 '5913',
 '5947',
 '5962',
 '5966',
 '5968',
 '6033',
 '6044',
 '6046',
 '6052',
 '

In [19]:
cons_txn_merged = cons_df.merge(txn_avg_monthly, on='prism_consumer_id', how='left')

In [57]:
train_df = train_df[['prism_consumer_id', 'DQ_TARGET']]

In [122]:
#txn_df['year_month'] = pd.to_datetime(txn_df['posted_date']).dt.to_period('M')
acct_df['year_month'] = pd.to_datetime(acct_df['balance_date']).dt.to_period('M')
acct_df

,prism_consumer_id,prism_account_id,account_type,balance_date,balance,year_month
0,3023,0,SAVINGS,2021-08-31,90.57,2021-08
1,3023,1,CHECKING,2021-08-31,225.95,2021-08
2,4416,2,SAVINGS,2022-03-31,15157.17,2022-03
3,4416,3,CHECKING,2022-03-31,66.42,2022-03
4,4227,4,CHECKING,2021-07-31,7042.90,2021-07
...,...,...,...,...,...,...
24461,11500,24461,CHECKING,2022-03-27,732.75,2022-03
24462,11615,24462,SAVINGS,2022-03-30,5.00,2022-03
24463,11615,24463,CHECKING,2022-03-30,1956.46,2022-03
24464,12210,24464,CHECKING,2022-03-28,2701.51,2022-03


In [123]:
txn_df

,prism_consumer_id,prism_transaction_id,category,amount,credit_or_debit,posted_date,year_month
0,3023,0,4,0.05,CREDIT,2021-04-16,2021-04
1,3023,1,12,481.56,CREDIT,2021-04-30,2021-04
2,3023,2,4,0.05,CREDIT,2021-05-16,2021-05
3,3023,3,4,0.07,CREDIT,2021-06-16,2021-06
4,3023,4,4,0.06,CREDIT,2021-07-16,2021-07
...,...,...,...,...,...,...,...
6407316,10533,6405304,31,4.96,DEBIT,2022-03-11,2022-03
6407317,10533,6405305,12,63.48,DEBIT,2022-03-30,2022-03
6407318,10533,6405306,12,53.99,DEBIT,2022-03-30,2022-03
6407319,10533,6405307,12,175.98,DEBIT,2022-03-31,2022-03


# Model Training

In [89]:
# Define a function to train Logistic Regression on a dataframe
def train_logistic_regression(df):
    """
    Train a logistic regression model using all columns (except the first) to predict the first column.

    Args:
        df (pd.DataFrame): Input dataframe where the first column (excluding index) is the target variable.

    Returns:
        None: Prints model evaluation metrics.
    """
    # Ensure the dataframe has at least two columns
    if df.shape[1] < 2:
        raise ValueError("The dataframe must have at least two columns.")
    
    temp = train_df.merge(df, on='prism_consumer_id', how='left')
    temp = temp.dropna()
    temp = temp.drop(columns=['prism_consumer_id'])

    # Separate features (X) and target (y)
    y = temp['DQ_TARGET']
    X = temp.drop(columns=['DQ_TARGET'])

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)

    # Train Logistic Regression
    clf = LogisticRegression(random_state=0, max_iter=1000, class_weight="balanced")
    clf.fit(X_train, y_train)

    # Predict probabilities and classes
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_proba)

    print("Accuracy:", accuracy)
    print("F1 Score:", f1)
    print("AUC-ROC Score:", auc)
    print("Classification Report:\n", classification_report(y_test, y_pred))

In [85]:
# Normalized Average Net Monthly Inflow

train_logistic_regression(txn_avg_monthly[['prism_consumer_id', 'average_nmi_z_score']])

Accuracy: 0.9171523730224813
F1 Score: 0.8779169729170678
AUC-ROC Score: 0.5656485453445526
Classification Report:
               precision    recall  f1-score   support

         0.0       0.92      1.00      0.96      2204
         1.0       0.00      0.00      0.00       198

    accuracy                           0.92      2402
   macro avg       0.46      0.50      0.48      2402
weighted avg       0.84      0.92      0.88      2402



In [92]:
# Normalized balance
acct_balance_df = acct_df.groupby("prism_consumer_id").agg(balance_sum = ('balance', 'sum')).reset_index()

zscoring_normalize(acct_balance_df, 'balance_sum')

train_logistic_regression(acct_balance_df[['prism_consumer_id', 'balance_sum_z_score']])

Accuracy: 0.3793254489706526
F1 Score: 0.46760211409791264
AUC-ROC Score: 0.7389294932217358
Classification Report:
               precision    recall  f1-score   support

         0.0       0.98      0.33      0.49      2089
         1.0       0.11      0.92      0.20       194

    accuracy                           0.38      2283
   macro avg       0.55      0.63      0.35      2283
weighted avg       0.91      0.38      0.47      2283



In [118]:
acct_df[acct_df["prism_consumer_id"] == '11615']

,prism_consumer_id,prism_account_id,account_type,balance_date,balance
24462,11615,24462,SAVINGS,2022-03-30,5.00
24463,11615,24463,CHECKING,2022-03-30,1956.46
24465,11615,24465,CHECKING,2022-03-30,7967.45


In [115]:
acct_df["prism_consumer_id"][0]

'3023'

In [117]:
acct_df["prism_consumer_id"]

0         3023
1         3023
2         4416
3         4416
4         4227
         ...  
24461    11500
24462    11615
24463    11615
24464    12210
24465    11615
Name: prism_consumer_id, Length: 24466, dtype: object